In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import norm
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

import joblib

In [8]:
BINS   = [0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100]
LABELS = ["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]

def bucket_mae(df, preds):
    """MAE by moneyness bucket for each model; last row = all options."""
    t = pd.DataFrame({"bucket": pd.cut(df["moneyness (S/K)"], bins=BINS, labels=LABELS)})
    for k, p in preds.items():
        t[k] = np.abs(df["price"].values - np.asarray(p))
    out = t.groupby("bucket", observed=True).mean()
    out.insert(0, "n", t.groupby("bucket", observed=True).size())
    out.loc["ALL"] = [len(t)] + list(t.drop(columns="bucket").mean())
    return out.astype({"n": int})

### Load data (same preprocessing/split as modeling_spx.ipynb)

In [9]:
spx = pd.read_csv("data/processed/SPX_cleaned.csv")
spx["date"] = pd.to_datetime(spx["date"])
spx["exdate"] = pd.to_datetime(spx["exdate"])

print(spx.shape)
spx.head()


(7688150, 17)


,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price,volatility,rate,days_to_maturity,T,q
0,108105,2020-08-31,2020-09-18,100.0,3402.5,3408.2,0,611,133485146,3500.31,35.003100,3405.35,0.111899,0.002148,18,0.049315,0.01589
1,108105,2020-08-31,2020-09-18,1000.0,2503.2,2508.0,0,37663,130915017,3500.31,3.500310,2505.60,0.111899,0.002148,18,0.049315,0.01589
2,108105,2020-08-31,2020-09-18,1100.0,2403.4,2407.9,0,96,130915018,3500.31,3.182100,2405.65,0.111899,0.002148,18,0.049315,0.01589
3,108105,2020-08-31,2020-09-18,1200.0,2303.5,2307.9,0,19,129372797,3500.31,2.916925,2305.70,0.111899,0.002148,18,0.049315,0.01589
4,108105,2020-08-31,2020-09-18,1250.0,2253.7,2258.0,0,20,129372798,3500.31,2.800248,2255.85,0.111899,0.002148,18,0.049315,0.01589


In [10]:
spx = spx.sort_values("date").reset_index(drop=True)

train_cutoff = spx["date"].quantile(0.70)
val_cutoff = spx["date"].quantile(0.85)

train = spx[spx["date"] <= train_cutoff]
val = spx[(spx["date"] > train_cutoff) & (spx["date"] <= val_cutoff)]
test = spx[spx["date"] > val_cutoff]

print(f"Train: {len(train)} rows, {train['date'].min()} to {train['date'].max()}")
print(f"Validation: {len(val)} rows, {val['date'].min()} to {val['date'].max()}")
print(f"Test: {len(test)} rows, {test['date'].min()} to {test['date'].max()}")


Train: 5388081 rows, 2020-08-31 00:00:00 to 2024-06-04 00:00:00
Validation: 1149725 rows, 2024-06-05 00:00:00 to 2025-01-24 00:00:00
Test: 1150344 rows, 2025-01-27 00:00:00 to 2025-08-29 00:00:00


# Ordered notebook

## 1.1 Experiment (v1): 6 features + raw target - Baseline

In [17]:
# ============================================================
# Neural Network — 6 features, raw target (price) — AAPL
# ============================================================

feature_cols_1 = ["moneyness (S/K)", "strike_price", "T", "rate", "q", "volatility"]
target_col = "price"

X_train_1 = train[feature_cols_1].values
X_val_1 = val[feature_cols_1].values
y_train_1 = train[target_col].values.reshape(-1, 1)
y_val = val[target_col].values

scaler_1 = StandardScaler().fit(X_train_1)
X_train_1_scaled = scaler_1.transform(X_train_1)
X_val_1_scaled = scaler_1.transform(X_val_1)

y_scaler_1 = StandardScaler().fit(y_train_1)      # the raw price needs scaling too
y_train_1_scaled = y_scaler_1.transform(y_train_1).ravel()

selection_1 = []

In [ ]:
for n_units in range(1, 15):
    model = MLPRegressor(
        hidden_layer_sizes=(n_units,), # only one layer, tuning nummber of neurons
        activation="logistic", # common choice in the literature
        solver="adam",
        max_iter=2000, # max num but there is early stopping
        batch_size=2000, # default
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42
    )
    model.fit(X_train_1_scaled, y_train_1_scaled)
    val_pred = y_scaler_1.inverse_transform(model.predict(X_val_1_scaled).reshape(-1, 1)).ravel()
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"n_units={n_units}: val_rmse={val_rmse:.4f}, n_iter={model.n_iter_}, hit_max={model.n_iter_ >= 2000}")
    selection_1.append({"n_units": n_units, "val_rmse": val_rmse})

selection_1 = pd.DataFrame(selection_1)

n_units=1: val_rmse=272.8133, n_iter=29, hit_max=False


c:\Users\glmar\OneDrive\Bemacs Bocconi\TESI\Thesis\Priced-by-intelligence\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


n_units=2: val_rmse=209.1933, n_iter=16, hit_max=False


In [10]:
best_n_1 = 13   # chosen from the selection table above

model_1 = MLPRegressor(
    hidden_layer_sizes=(best_n_1,),
    activation="logistic",
    solver="adam",
    max_iter=2000,
    batch_size=2000,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
model_1.fit(X_train_1_scaled, y_train_1_scaled)

nn_pred_1_train = y_scaler_1.inverse_transform(model_1.predict(X_train_1_scaled).reshape(-1, 1)).ravel()
nn_pred_1 = y_scaler_1.inverse_transform(model_1.predict(X_val_1_scaled).reshape(-1, 1)).ravel()

y_train_price = train[target_col].values

print(f"Neural Network (6 features, raw target, {best_n_1} hidden units) — AAPL")
print(f"Train — RMSE={np.sqrt(mean_squared_error(y_train_price, nn_pred_1_train)):.4f}  "
      f"MAE={mean_absolute_error(y_train_price, nn_pred_1_train):.4f}")
print(f"Val   — RMSE={np.sqrt(mean_squared_error(y_val, nn_pred_1)):.4f}  "
      f"MAE={mean_absolute_error(y_val, nn_pred_1):.4f}  neg={(nn_pred_1 < 0).sum()}")

Neural Network (6 features, raw target, 13 hidden units) — AAPL
Train — RMSE=27.5697  MAE=16.4768
Val   — RMSE=101.6885  MAE=57.1557  neg=91650


In [11]:
train_bucket_1 = bucket_mae(train, {"v1": nn_pred_1_train})
val_bucket_1   = bucket_mae(val,   {"v1": nn_pred_1})

overfit_1 = pd.DataFrame({
    "n_train":   train_bucket_1["n"],
    "train_mae": train_bucket_1["v1"],
    "n_val":     val_bucket_1["n"],
    "val_mae":   val_bucket_1["v1"],
})
overfit_1["ratio_val_vs_train"] = overfit_1["val_mae"] / overfit_1["train_mae"]
overfit_1.round(3)

,n_train,train_mae,n_val,val_mae,ratio_val_vs_train
bucket,,,,,
Deep OTM,155781,15.967,21332,58.706,3.677
OTM,1035010,14.731,157667,30.212,2.051
ATM,2184122,15.283,529060,37.039,2.424
ITM,1250438,12.004,272607,48.811,4.066
Deep ITM,587889,18.349,143006,110.904,6.044
Very Deep ITM,130570,35.851,17812,264.283,7.372
Extreme ITM,44271,162.298,8241,755.746,4.657
ALL,5388081,16.477,1149725,57.156,3.469


## 1.2 Experiment (v2): moneyness(S/K) + K + log transformed target

In [8]:
# ============================================================
# Neural Network — 6 features, log(C+1) target — AAPL
# ============================================================

# same inputs as v1, so the scaler can be reused
y_train_2 = np.log1p(train[target_col].values)

selection_2 = []

In [ ]:
# ============================================================
# Neural Network — 6 features, log(C+1) target — AAPL
# ============================================================

# same inputs as v1, so the scaler can be reused
y_train_2 = np.log1p(train[target_col].values)

selection_2 = []

for n_units in range(1, 16):
    model = MLPRegressor(
        hidden_layer_sizes=(n_units,),
        activation="logistic",
        solver="adam",
        max_iter=2000,
        batch_size=2000,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42
    )
    model.fit(X_train_1_scaled, y_train_2)
    val_pred = np.expm1(model.predict(X_val_1_scaled))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"n_units={n_units}: val_rmse={val_rmse:.4f}, n_iter={model.n_iter_}, hit_max={model.n_iter_ >= 2000}")
    selection_2.append({"n_units": n_units, "val_rmse": val_rmse})

selection_2 = pd.DataFrame(selection_2)


n_units=1: val_rmse=572.7733, n_iter=34, hit_max=False
n_units=2: val_rmse=398.3112, n_iter=59, hit_max=False
n_units=3: val_rmse=341.2683, n_iter=66, hit_max=False
n_units=4: val_rmse=293.7427, n_iter=55, hit_max=False
n_units=5: val_rmse=341.2722, n_iter=64, hit_max=False
n_units=6: val_rmse=240.1220, n_iter=88, hit_max=False
n_units=7: val_rmse=299.2283, n_iter=72, hit_max=False
n_units=8: val_rmse=255.8164, n_iter=66, hit_max=False
n_units=9: val_rmse=256.3920, n_iter=70, hit_max=False
n_units=10: val_rmse=273.4472, n_iter=65, hit_max=False
n_units=11: val_rmse=225.6530, n_iter=61, hit_max=False
n_units=12: val_rmse=288.6676, n_iter=57, hit_max=False
n_units=13: val_rmse=290.9678, n_iter=60, hit_max=False
n_units=14: val_rmse=242.0777, n_iter=76, hit_max=False
n_units=15: val_rmse=232.7351, n_iter=60, hit_max=False


: 

In [12]:
# Fit with the selected number of units and evaluate on train and validation
best_n_2 = 9

model_2 = MLPRegressor(
    hidden_layer_sizes=(best_n_2,),
    activation="logistic",
    solver="adam",
    max_iter=2000,
    batch_size=2000,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
model_2.fit(X_train_1_scaled, y_train_2)

nn_pred_2_train = np.expm1(model_2.predict(X_train_1_scaled))
nn_pred_2 = np.expm1(model_2.predict(X_val_1_scaled))

print(f"v2 Train — RMSE={np.sqrt(mean_squared_error(y_train_price, nn_pred_2_train)):.4f}  "
      f"MAE={mean_absolute_error(y_train_price, nn_pred_2_train):.4f}")
print(f"v2 Val   — RMSE={np.sqrt(mean_squared_error(y_val, nn_pred_2)):.4f}  "
      f"MAE={mean_absolute_error(y_val, nn_pred_2):.4f}  neg={(nn_pred_2 < 0).sum()}")

v2 Train — RMSE=134.1795  MAE=47.0684
v2 Val   — RMSE=256.3920  MAE=121.0275  neg=18392


In [3]:
# MAE by moneyness bucket — train vs validation
train_bucket_2 = bucket_mae(train, {"v2": nn_pred_2_train})
val_bucket_2   = bucket_mae(val,   {"v2": nn_pred_2})

overfit_2 = pd.DataFrame({
    "n_train":   train_bucket_2["n"],
    "train_mae": train_bucket_2["v2"],
    "n_val":     val_bucket_2["n"],
    "bs_mae":    val_bucket_2["BS"],
    "val_mae":   val_bucket_2["v2"],
})
overfit_2["ratio_val_vs_train"] = overfit_2["val_mae"] / overfit_2["train_mae"]
overfit_2.round(3)

NameError: name 'train' is not defined

## 1.3 Experiment (v3): 5 inputs + normalized output

In [ ]:
# ============================================================
# Neural Network — 5 features, normalized target (price/K) — AAPL
# ============================================================

feature_cols_3 = ["moneyness (S/K)", "T", "rate", "q", "volatility"]

X_train_3 = train[feature_cols_3].values
X_val_3 = val[feature_cols_3].values

K_train = train["strike_price"].values
K_val = val["strike_price"].values

y_train_3 = (train[target_col].values / K_train).reshape(-1, 1)

scaler_3 = StandardScaler().fit(X_train_3)
X_train_3_scaled = scaler_3.transform(X_train_3)
X_val_3_scaled = scaler_3.transform(X_val_3)

y_scaler_3 = StandardScaler().fit(y_train_3)      # C/K is small, scaling keeps the optimiser stable
y_train_3_scaled = y_scaler_3.transform(y_train_3).ravel()

selection_3 = []

for n_units in range(1, 16):
    model = MLPRegressor(
        hidden_layer_sizes=(n_units,),
        activation="logistic",
        solver="adam",
        max_iter=2000,
        batch_size=2000,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42
    )
    model.fit(X_train_3_scaled, y_train_3_scaled)
    val_pred_ratio = y_scaler_3.inverse_transform(model.predict(X_val_3_scaled).reshape(-1, 1)).ravel()
    val_pred = val_pred_ratio * K_val
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"n_units={n_units}: val_rmse={val_rmse:.4f}, n_iter={model.n_iter_}, hit_max={model.n_iter_ >= 2000}")
    selection_3.append({"n_units": n_units, "val_rmse": val_rmse})

selection_3 = pd.DataFrame(selection_3)
best_n_3 = int(selection_3.loc[selection_3["val_rmse"].idxmin(), "n_units"])
print("Selected hidden units:", best_n_3)

In [ ]:
# Fit with the selected number of units and evaluate on train and validation
model_3 = MLPRegressor(
    hidden_layer_sizes=(best_n_3,),
    activation="logistic",
    solver="adam",
    max_iter=2000,
    batch_size=2000,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
model_3.fit(X_train_3_scaled, y_train_3_scaled)

nn_pred_3_train = y_scaler_3.inverse_transform(
    model_3.predict(X_train_3_scaled).reshape(-1, 1)).ravel() * K_train
nn_pred_3 = y_scaler_3.inverse_transform(
    model_3.predict(X_val_3_scaled).reshape(-1, 1)).ravel() * K_val

print(f"v3 Train — RMSE={np.sqrt(mean_squared_error(y_train_price, nn_pred_3_train)):.4f}  "
      f"MAE={mean_absolute_error(y_train_price, nn_pred_3_train):.4f}")
print(f"v3 Val   — RMSE={np.sqrt(mean_squared_error(y_val, nn_pred_3)):.4f}  "
      f"MAE={mean_absolute_error(y_val, nn_pred_3):.4f}  neg={(nn_pred_3 < 0).sum()}")

In [ ]:
# MAE by moneyness bucket — train vs validation
train_bucket_3 = bucket_mae(train, {"v3": nn_pred_3_train})
val_bucket_3   = bucket_mae(val,   {"v3": nn_pred_3})

overfit_3 = pd.DataFrame({
    "n_train":   train_bucket_3["n"],
    "train_mae": train_bucket_3["v3"],
    "n_val":     val_bucket_3["n"],
    "bs_mae":    val_bucket_3["BS"],
    "val_mae":   val_bucket_3["v3"],
})
overfit_3["ratio_val_vs_train"] = overfit_3["val_mae"] / overfit_3["train_mae"]
overfit_3.round(3)

## 1.4 Experiment (v4): 5 inputs + log-transformed output

In [ ]:
# ============================================================
# Neural Network — 5 features, log(C/K) target — AAPL
# ============================================================

# same inputs as v3, so the scaler can be reused
y_train_4 = np.log(train[target_col].values / K_train)

selection_4 = []

for n_units in range(1, 16):
    model = MLPRegressor(
        hidden_layer_sizes=(n_units,),
        activation="logistic",
        solver="adam",
        max_iter=2000,
        batch_size=2000,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42
    )
    model.fit(X_train_3_scaled, y_train_4)
    val_pred = np.exp(model.predict(X_val_3_scaled)) * K_val
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"n_units={n_units}: val_rmse={val_rmse:.4f}, n_iter={model.n_iter_}, hit_max={model.n_iter_ >= 2000}")
    selection_4.append({"n_units": n_units, "val_rmse": val_rmse})

selection_4 = pd.DataFrame(selection_4)
selection_4

In [ ]:
best_n_4 = 10   # chosen from the selection table above

model_4 = MLPRegressor(
    hidden_layer_sizes=(best_n_4,),
    activation="logistic",
    solver="adam",
    max_iter=2000,
    batch_size=2000,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
model_4.fit(X_train_3_scaled, y_train_4)

nn_pred_4_train = np.exp(model_4.predict(X_train_3_scaled)) * K_train
nn_pred_4 = np.exp(model_4.predict(X_val_3_scaled)) * K_val

print(f"Neural Network (5 features, log(C/K), {best_n_4} hidden units) — AAPL")
print(f"Train — RMSE={np.sqrt(mean_squared_error(y_train_price, nn_pred_4_train)):.4f}  "
      f"MAE={mean_absolute_error(y_train_price, nn_pred_4_train):.4f}")
print(f"Val   — RMSE={np.sqrt(mean_squared_error(y_val, nn_pred_4)):.4f}  "
      f"MAE={mean_absolute_error(y_val, nn_pred_4):.4f}  neg={(nn_pred_4 < 0).sum()}")

In [ ]:
train_bucket_4 = bucket_mae(train, {"v4": nn_pred_4_train})
val_bucket_4   = bucket_mae(val,   {"v4": nn_pred_4})

overfit_4 = pd.DataFrame({
    "n_train":   train_bucket_4["n"],
    "train_mae": train_bucket_4["v4"],
    "n_val":     val_bucket_4["n"],
    "bs_mae":    val_bucket_4["BS"],
    "val_mae":   val_bucket_4["v4"],
})
overfit_4["ratio_val_vs_train"] = overfit_4["val_mae"] / overfit_4["train_mae"]
overfit_4.round(3)

### 1.5 Comparison of the four configurations on validation set

In [ ]:
bucket_mae(val, {"v1 C (6f)":      nn_pred_1,
                 "v2 log1pC (6f)": nn_pred_2,
                 "v3 C/K (5f)":    nn_pred_3,
                 "v4 logC/K (5f)": nn_pred_4}).round(3)

## 2. Final model: retrain on train+val and test

# Old notebook

## Neural Network — target normalized by strike (price/K), 6 features — SPX

Directly using the winning AAPL formulation: input `moneyness (S/K)`, `strike_price`, `T`, `rate`, `q`, `volatility`; target `price/strike_price` (log-transformed), predictions multiplied back by `strike_price` before evaluation.

In [8]:
# ============================================================
# Neural Network — target normalized by strike (price/K), 6 features — SPX
# ============================================================

feature_cols_nn6 = ["moneyness (S/K)", "strike_price", "T", "rate", "q", "volatility"]

X_train_full6 = train[feature_cols_nn6].values
y_train_ratio = train["price"].values / train["strike_price"].values
y_train_ratio_log = np.log1p(y_train_ratio)

X_val6 = val[feature_cols_nn6].values
y_val_ratio = val["price"].values / val["strike_price"].values
y_val_price = val["price"].values   # keep raw price for RMSE evaluation

scaler6b = StandardScaler()
X_train_full6_scaled = scaler6b.fit_transform(X_train_full6)
X_val6_scaled = scaler6b.transform(X_val6)

selection_results6b = []

for n_units in range(1, 16):
    model = MLPRegressor(
        hidden_layer_sizes=(n_units,),
        activation="logistic",
        solver="adam",
        max_iter=2000,
        batch_size=2000,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42
    )
    model.fit(X_train_full6_scaled, y_train_ratio_log)
    val_pred_ratio_log = model.predict(X_val6_scaled)
    val_pred_ratio = np.expm1(val_pred_ratio_log)
    val_pred_price = val_pred_ratio * val["strike_price"].values   # multiply back by K
    val_rmse = np.sqrt(mean_squared_error(y_val_price, val_pred_price))
    print(f"n_units={n_units}: val_rmse={val_rmse:.4f}, n_iter={model.n_iter_}, hit_max={model.n_iter_ >= 2000}")
    selection_results6b.append({"n_units": n_units, "val_rmse": val_rmse})



n_units=1: val_rmse=583.7380, n_iter=21, hit_max=False
n_units=2: val_rmse=232.5865, n_iter=27, hit_max=False
n_units=3: val_rmse=153.9476, n_iter=22, hit_max=False
n_units=4: val_rmse=88.2412, n_iter=27, hit_max=False
n_units=5: val_rmse=116.0537, n_iter=25, hit_max=False
n_units=6: val_rmse=89.5544, n_iter=34, hit_max=False
n_units=7: val_rmse=91.9247, n_iter=26, hit_max=False
n_units=8: val_rmse=106.9189, n_iter=27, hit_max=False
n_units=9: val_rmse=93.7448, n_iter=25, hit_max=False
n_units=10: val_rmse=90.6851, n_iter=27, hit_max=False
n_units=11: val_rmse=103.7590, n_iter=24, hit_max=False
n_units=12: val_rmse=82.5409, n_iter=25, hit_max=False
n_units=13: val_rmse=111.2946, n_iter=25, hit_max=False
n_units=14: val_rmse=83.5985, n_iter=25, hit_max=False
n_units=15: val_rmse=90.7306, n_iter=25, hit_max=False

Best hidden units (lowest val RMSE): 12


**After running the cell above:** apply the elbow-method check (like for AAPL) before trusting the nominal winner — look for where the marginal RMSE improvement becomes small/non-monotonic, and set `best_n_units6b` manually below to that value instead of blindly taking the minimum.

In [16]:
# ============================================================
# Final Neural Network: retrain on train+val, evaluate once on test — SPX
# ============================================================

best_n_units6b = 10  

train_val = pd.concat([train, val], ignore_index=True)

X_train_val6 = train_val[feature_cols_nn6].values
y_train_val_ratio = train_val["price"].values / train_val["strike_price"].values
y_train_val_ratio_log = np.log1p(y_train_val_ratio)

X_test6 = test[feature_cols_nn6].values
y_test_price = test["price"].values

scaler6b_final = StandardScaler()
X_train_val6_scaled = scaler6b_final.fit_transform(X_train_val6)
X_test6_scaled = scaler6b_final.transform(X_test6)

nn_model6b = MLPRegressor(
    hidden_layer_sizes=(best_n_units6b,),
    activation="logistic",
    solver="adam",
    max_iter=2000,
    batch_size=2000,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
nn_model6b.fit(X_train_val6_scaled, y_train_val_ratio_log)

y_pred_ratio_log = nn_model6b.predict(X_test6_scaled)
y_pred_ratio = np.expm1(y_pred_ratio_log)
y_pred_nn6b = y_pred_ratio * test["strike_price"].values

nn6b_rmse = np.sqrt(mean_squared_error(y_test_price, y_pred_nn6b))
nn6b_mae = mean_absolute_error(y_test_price, y_pred_nn6b)

print(f"Neural Network (normalized target, {best_n_units6b} hidden units) — SPX Test set")
print(f"RMSE: {nn6b_rmse:.4f}")
print(f"MAE:  {nn6b_mae:.4f}")


Neural Network (normalized target, 10 hidden units) — SPX Test set
RMSE: 58.1196
MAE:  42.3406


In [17]:
# Train vs test error — SPX Neural Network (normalized target)
y_pred_train_ratio = np.expm1(nn_model6b.predict(X_train_val6_scaled))
y_pred_train_price = y_pred_train_ratio * train_val["strike_price"].values
y_train_val_price = train_val["price"].values

train_rmse_nn6b = np.sqrt(mean_squared_error(y_train_val_price, y_pred_train_price))
train_mae_nn6b = mean_absolute_error(y_train_val_price, y_pred_train_price)

print(f"SPX NN (normalized target) — Train (train+val): RMSE={train_rmse_nn6b:.4f}, MAE={train_mae_nn6b:.4f}")
print(f"SPX NN (normalized target) — Test:              RMSE={nn6b_rmse:.4f}, MAE={nn6b_mae:.4f}")
print(f"Ratio test/train RMSE: {nn6b_rmse/train_rmse_nn6b:.2f}x")
print(f"Ratio test/train MAE:  {nn6b_mae/train_mae_nn6b:.2f}x")

SPX NN (normalized target) — Train (train+val): RMSE=36.9420, MAE=27.2220
SPX NN (normalized target) — Test:              RMSE=58.1196, MAE=42.3406
Ratio test/train RMSE: 1.57x
Ratio test/train MAE:  1.56x


In [12]:
# ============================================================
# Neural Network error by moneyness bucket — SPX test set
# ============================================================

nn6b_check = test.copy()
nn6b_check["nn6b_pred"] = y_pred_nn6b
nn6b_check["nn6b_error_abs"] = np.abs(nn6b_check["price"] - nn6b_check["nn6b_pred"])
nn6b_check["nn6b_error_pct"] = (nn6b_check["nn6b_error_abs"] / nn6b_check["price"]) * 100

nn6b_check["moneyness_bucket"] = pd.cut(
    nn6b_check["moneyness (S/K)"],
    bins=[0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100],
    labels=["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]
)

nn6b_bucket_summary = nn6b_check.groupby("moneyness_bucket").agg(
    n_options=("nn6b_error_abs", "size"),
    nn6b_mae=("nn6b_error_abs", "mean"),
    nn6b_mape=("nn6b_error_pct", "mean"),
)
nn6b_bucket_summary["pct_of_test"] = (nn6b_bucket_summary["n_options"] / nn6b_bucket_summary["n_options"].sum()) * 100

nn6b_bucket_summary


,n_options,nn6b_mae,nn6b_mape,pct_of_test
moneyness_bucket,,,,
Deep OTM,30051,55.806007,20305.749745,2.612349
OTM,227962,58.739265,9020.691974,19.816855
ATM,490931,54.808740,1802.683352,42.676886
ITM,251452,130.540533,20.619952,21.858853
Deep ITM,123621,70.007380,4.916155,10.746438
Very Deep ITM,17799,66.004504,1.759046,1.547276
Extreme ITM,8528,143.579527,2.701804,0.741343


**Next:** bring in `bs_mae` per bucket (from `modeling_spx.ipynb`, cell producing `bucket_summary`) and `xgb_mae` (the tuned final model) to build the same 3/4-way comparison table (BS vs XGBoost vs NN) as for AAPL.

# ANOTHER ATTEMPT - target normalized

In [5]:
# ============================================================
# Final Neural Network: retrain on train+val, evaluate once on test — SPX
# ============================================================

feature_cols_nn6 = ["moneyness (S/K)", "T", "rate", "q", "volatility"]

best_n_units6b = 15  # placeholder — replace with your elbow-method choice from the selection above

train_val = pd.concat([train, val], ignore_index=True)

X_train_val6 = train_val[feature_cols_nn6].values
y_train_val_ratio = train_val["price"].values / train_val["strike_price"].values
y_train_val_ratio_log = np.log1p(y_train_val_ratio)

X_test6 = test[feature_cols_nn6].values
y_test_price = test["price"].values

scaler6b_final = StandardScaler()
X_train_val6_scaled = scaler6b_final.fit_transform(X_train_val6)
X_test6_scaled = scaler6b_final.transform(X_test6)

nn_model6b = MLPRegressor(
    hidden_layer_sizes=(best_n_units6b,),
    activation="logistic",
    solver="adam",
    max_iter=2000,
    batch_size=2000,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
nn_model6b.fit(X_train_val6_scaled, y_train_val_ratio_log)

y_pred_ratio_log = nn_model6b.predict(X_test6_scaled)
y_pred_ratio = np.expm1(y_pred_ratio_log)
y_pred_nn6b = y_pred_ratio * test["strike_price"].values

nn6b_rmse = np.sqrt(mean_squared_error(y_test_price, y_pred_nn6b))
nn6b_mae = mean_absolute_error(y_test_price, y_pred_nn6b)

print(f"Neural Network (normalized target, {best_n_units6b} hidden units) — SPX Test set")
print(f"RMSE: {nn6b_rmse:.4f}")
print(f"MAE:  {nn6b_mae:.4f}")


Neural Network (normalized target, 15 hidden units) — SPX Test set
RMSE: 50.2669
MAE:  37.6775


In [6]:
# ============================================================
# Neural Network error by moneyness bucket — SPX test set
# ============================================================

nn6b_check = test.copy()
nn6b_check["nn6b_pred"] = y_pred_nn6b
nn6b_check["nn6b_error_abs"] = np.abs(nn6b_check["price"] - nn6b_check["nn6b_pred"])
nn6b_check["nn6b_error_pct"] = (nn6b_check["nn6b_error_abs"] / nn6b_check["price"]) * 100

nn6b_check["moneyness_bucket"] = pd.cut(
    nn6b_check["moneyness (S/K)"],
    bins=[0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100],
    labels=["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]
)

nn6b_bucket_summary = nn6b_check.groupby("moneyness_bucket").agg(
    n_options=("nn6b_error_abs", "size"),
    nn6b_mae=("nn6b_error_abs", "mean"),
    nn6b_mape=("nn6b_error_pct", "mean"),
)
nn6b_bucket_summary["pct_of_test"] = (nn6b_bucket_summary["n_options"] / nn6b_bucket_summary["n_options"].sum()) * 100

nn6b_bucket_summary


,n_options,nn6b_mae,nn6b_mape,pct_of_test
moneyness_bucket,,,,
Deep OTM,30051,66.857271,30463.987418,2.612349
OTM,227962,54.490119,16215.087902,19.816855
ATM,490931,27.737690,663.908832,42.676886
ITM,251452,41.122673,7.303134,21.858853
Deep ITM,123621,30.524991,1.981901,10.746438
Very Deep ITM,17799,39.420709,1.076888,1.547276
Extreme ITM,8528,56.099796,1.051990,0.741343


Training error

In [8]:
# Errore sul TRAINING set (train+val, gli stessi dati usati per allenare)
y_pred_train_ratio_log = nn_model6b.predict(X_train_val6_scaled)
y_pred_train_ratio = np.expm1(y_pred_train_ratio_log)
y_pred_train_price = y_pred_train_ratio * train_val["strike_price"].values
y_train_val_price = train_val["price"].values

train_rmse = np.sqrt(mean_squared_error(y_train_val_price, y_pred_train_price))
train_mae = mean_absolute_error(y_train_val_price, y_pred_train_price)

print(f"Training set (train+val) — RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}")
print(f"Test set                  — RMSE: {nn6b_rmse:.4f}, MAE: {nn6b_mae:.4f}")
print(f"\nRatio test/train RMSE: {nn6b_rmse/train_rmse:.2f}x")

Training set (train+val) — RMSE: 27.2876, MAE: 20.7672
Test set                  — RMSE: 50.2669, MAE: 37.6775

Ratio test/train RMSE: 1.84x


let's try to reduce overfitting with alpha regularization

In [9]:
for alpha_test in [0.0001, 0.01]:
    model = MLPRegressor(
        hidden_layer_sizes=(15,),
        activation="logistic",
        solver="adam",
        alpha=alpha_test,
        max_iter=2000,
        batch_size=2000,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42
    )
    model.fit(X_train_val6_scaled, y_train_val_ratio_log)
    pred_ratio = np.expm1(model.predict(X_test6_scaled))
    pred_price = pred_ratio * test["strike_price"].values
    test_rmse = np.sqrt(mean_squared_error(test["price"].values, pred_price))
    print(f"alpha={alpha_test}: test_rmse={test_rmse:.4f}")

alpha=0.0001: test_rmse=50.2669
alpha=0.01: test_rmse=119.9857


## Save data

In [10]:
print([v for v in dir() if any(k in v.lower() for k in ['model', 'mlp', 'scaler', 'feature_cols'])])

['MLPRegressor', 'StandardScaler', 'feature_cols_nn6', 'model', 'nn_model6b', 'scaler6b_final']


In [11]:
print("features:", feature_cols_nn6)
print("n features expected by scaler:", scaler6b_final.n_features_in_)
print("hidden units:", nn_model6b.hidden_layer_sizes)

features: ['moneyness (S/K)', 'T', 'rate', 'q', 'volatility']
n features expected by scaler: 5
hidden units: (15,)


In [14]:
joblib.dump(nn_model6b, "nn_spx_model.pkl")
joblib.dump(scaler6b_final, "nn_spx_scaler.pkl")
print("saved nn spx")

saved nn spx


In [15]:
print(train[["T", "rate", "q", "volatility", "strike_price", "close"]].median())
print("\nmoneyness 99th pct:", train["moneyness (S/K)"].quantile(0.99))
print("moneyness max:", train["moneyness (S/K)"].max())

T                  0.134247
rate               0.028766
q                  0.013111
volatility         0.140084
strike_price    4170.000000
close           4274.040000
dtype: float64

moneyness 99th pct: 4.304425
moneyness max: 47.9656
